# ASG-WM / FaithCast — **Train All**

Full training pipeline -> **`train_results.zip`** (checkpoints + ASG labels + tables).

**Pipeline:** settings -> install -> access preflight -> GO/NO-GO -> download -> auto-label -> Tier-0 -> Tier-1 (VLM curriculum) -> Tier-2 -> zip.

Colab A100: `Runtime -> Run all`. Laptop/CPU smoke: keep `DATASET="synthetic"`. Gates: `RUN_PLAN.md`.

In [ ]:
# --- 1. Locate / clone the repo ----------------------------------------------
# On Colab: set ASGWM_REPO_URL to your git remote and this cell clones it.
# Locally: run this notebook from the repo root (the cell is a no-op).
import os, sys, subprocess, pathlib
REPO_URL = os.environ.get("ASGWM_REPO_URL", "")
REPO_DIR = "Forecasting-Through-Meteorological-Reasoning"
if not pathlib.Path("src/asgwm").exists():
    if not pathlib.Path(REPO_DIR, "src", "asgwm").exists():
        assert REPO_URL, "Set ASGWM_REPO_URL (Colab) or launch from the repo root."
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
assert pathlib.Path("src/asgwm").exists(), f"Run from repo root; cwd={os.getcwd()}"
print("repo root:", os.getcwd())


In [ ]:
# --- 2. Settings -------------------------------------------------------------
import subprocess, sys, itertools
DATASET      = "synthetic"   # "sevir" | "synthetic" | "nexrad" | "mrms"
REQUIRE_REAL = False         # True => a real-data load failure RAISES (use for paid runs)
N_EVENTS     = 64            # raise to 800+ for real runs (RUN_PLAN.md)
FIRST_RUN    = True          # time-boxed step caps that fit ~2 A100 sessions
CFG          = "src/configs/default.yaml"
print(f"dataset={DATASET}  require_real={REQUIRE_REAL}  n_events={N_EVENTS}  first_run={FIRST_RUN}")


In [ ]:
# --- 3. Install dependencies (core + the backends this DATASET needs) ---------
def _pip(*pkgs):
    if pkgs:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
EXTRAS = {
    "synthetic": [],
    "sevir":  ["s3fs", "h5py"],                       # SEVIR HDF5 on s3://sevir
    "nexrad": ["arm-pyart", "boto3"],                 # NEXRAD L2 polar read + gridding
    "mrms":   ["boto3", "xarray", "cfgrib", "eccodes"],  # MRMS gzipped GRIB2
}
_pip("-e", "./src")
_pip("-r", "src/requirements.txt")
_pip(*EXTRAS[DATASET])
print("installed core + extras for", DATASET, "->", EXTRAS[DATASET] or "(none)")


In [ ]:
# --- 4. Pipeline helpers -----------------------------------------------------
def ov(**kw):
    return list(itertools.chain.from_iterable(["--override", f"{k}={v}"] for k, v in kw.items()))
COMMON = ov(**{"data.dataset": DATASET, "data.require_real": str(REQUIRE_REAL).lower(),
               "data.n_train_events": N_EVENTS})
def run(script, extra=None):
    cmd = [sys.executable, f"src/scripts/{script}", "--config", CFG] + COMMON + (extra or [])
    print(">>", " ".join(cmd)); sys.stdout.flush(); subprocess.run(cmd, check=True)


In [ ]:
# --- 5. Data-access preflight (anonymous HTTPS; no heavy deps needed) ---------
# Confirms the bucket + (for nexrad/mrms) the configured case windows are reachable BEFORE
# the heavy download. Informational for synthetic.
if DATASET != "synthetic":
    subprocess.run([sys.executable, "datasets/check_access.py", "--datasets", DATASET], check=False)
else:
    print("synthetic: no download needed (offline generator).")


In [ ]:
# --- 6. Pre-flight GO/NO-GO (RUN_PLAN.md gates 1-2) --------------------------
run("99_preflight.py")


In [ ]:
# --- 7. Download / cache the dataset (idempotent) ---------------------------
run("00_download_data.py")


In [ ]:
# --- 8. Auto-label ASGs (CPU; cached once) ----------------------------------
run("01_autolabel.py")


In [ ]:
# --- 9. Tier-0: transition + deterministic renderer -------------------------
run("10_train_tier0.py", ov(**{"train.tier0.max_steps": 4000}) if FIRST_RUN else [])


In [ ]:
# --- 10. Tier-1: VLM five-phase curriculum (Ph-3 F1 gate) -------------------
run("20_train_tier1_curriculum.py", ov(**{"train.tier1.max_steps_per_phase": 4000}) if FIRST_RUN else [])


In [ ]:
# --- 11. Tier-2: end-to-end + intervention consistency ----------------------
run("30_train_tier2.py", ov(**{"train.tier2.max_steps": 12000}) if FIRST_RUN else [])


In [ ]:
# --- 12. Bundle checkpoints + labels + tables -> train_results.zip ------------
import glob, os, zipfile
OUT = "train_results.zip"
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
    for root in ["artifacts/ckpt", "artifacts/results", "artifacts/cache/asg"]:
        for p in glob.glob(os.path.join(root, "**", "*"), recursive=True):
            if os.path.isfile(p):
                z.write(p)
print(f"wrote {OUT}  ({os.path.getsize(OUT)/1e6:.1f} MB)" if os.path.exists(OUT) else "nothing to zip")


**Done.** Download `train_results.zip`, then run `eval.ipynb`.